# Resolving - Looking up what a citation actually says

Once a citation is parsed and normalised, it can be resolved against
the law corpus to retrieve the actual statutory text. This notebook
walks through the resolution process and what the results look like.

In [1]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".."], check=True)
sys.path.insert(0, str(__import__("pathlib").Path("..").resolve()))

from bundesrecht import Bundesrecht, normalise

## Load the corpus

Load `gesetze.jsonl` once. Expects a `token` file in the same directory
containing HuggingFace token on a single line.

In [2]:
from pathlib import Path
from huggingface_hub import hf_hub_download

token = Path("token").read_text().strip()

jsonl_path = hf_hub_download(
    repo_id="harshildarji/bundesrecht",
    filename="gesetze.jsonl",
    repo_type="dataset",
    token=token,
    local_dir=".",
)

lib = Bundesrecht(jsonl_path)
print(lib)

/Users/harshil/miniconda3/envs/berlin/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Bundesrecht(6867 laws loaded)


## A simple resolution

Query a paragraph reference and inspect the result.

In [3]:
r = lib.query("§ 433 Abs. 1 BGB")[0]
print(r.titel())

Vertragstypische Pflichten beim Kaufvertrag


In [4]:
print(r.full_text())

Durch den Kaufvertrag wird der Verkäufer einer Sache verpflichtet, dem Käufer die Sache zu übergeben und das Eigentum an der Sache zu verschaffen. Der Verkäufer hat dem Käufer die Sache frei von Sach- und Rechtsmängeln zu verschaffen.


In [5]:
print(r.resolved_depth)  # how deep did the resolver get?

absatz


## Resolved depth

The resolver goes down as far as the citation requests.
`resolved_depth` tells you what level was actually reached.

In [6]:
# section only - no sub-ref requested
r = lib.query("Art 49 GG")[0]
print(r.resolved_depth)

section


In [7]:
# absatz
r = lib.query("§ 433 Abs. 1 BGB")[0]
print(r.resolved_depth)

absatz


In [8]:
# nummer
r = lib.query("§ 2 Abs. 1 Nr. 1 UrhG")[0]
print(r.resolved_depth)

nummer


In [9]:
# buchstabe
r = lib.query("§ 81 Abs. 1 Nr. 1 Buchst. a BGB")[0]
print(r.resolved_depth)

buchstabe


## Partial resolution

When the requested depth cannot be reached - e.g. the Buchstabe doesn't
exist - the resolver falls back to the deepest level it could find
and explains via `resolution_note`.

In [10]:
r = lib.query("§ 2 Abs. 1 Nr. 1 Buchst. z UrhG")[0]
print(r.resolved_depth)

nummer


In [11]:
print(r.resolution_note)

Buchst. z not found in § 2 Abs. 1 Nr. 1 - resolved to Nr. 1


## Multi-target expansion

A citation referencing multiple sub-targets returns one `QueryResult` per target.

In [12]:
results = lib.query("§ 2 Abs. 1 Nr. 1, Nr. 7, Abs. 2 UrhG")
for r in results:
    print(str(r.reference))

§ 2 Abs. 1 Nr. 1 UrhG
§ 2 Abs. 1 Nr. 7 UrhG
§ 2 Abs. 2 UrhG


In [13]:
# each result resolves independently
for r in results:
    print(f"{str(r.reference)}: depth={r.resolved_depth}")

§ 2 Abs. 1 Nr. 1 UrhG: depth=nummer
§ 2 Abs. 1 Nr. 7 UrhG: depth=nummer
§ 2 Abs. 2 UrhG: depth=absatz


## Browsing a law directly

You can also access a law directly without querying a specific citation.

In [14]:
bgb = lib.get_law("BGB")
print(bgb.jurabk)
print(bgb.gesetze_id)

BGB
BGB::BJNR001950896


In [15]:
print(bgb.metadaten.get("langtitel"))

Bürgerliches Gesetzbuch


In [16]:
print(bgb.metadaten.get("ausfertigung_datum"))

1896-08-18


In [17]:
print(len(bgb.sections))

2541


In [18]:
sec = bgb.get_section("433")
print(sec["titel"])

Vertragstypische Pflichten beim Kaufvertrag


In [19]:
abs1 = bgb.get_absatz("433", 1)
print(abs1)

{'absatz': '(1) Durch den Kaufvertrag wird der Verkäufer einer Sache verpflichtet, dem Käufer die Sache zu übergeben und das Eigentum an der Sache zu verschaffen. Der Verkäufer hat dem Käufer die Sache frei von Sach- und Rechtsmängeln zu verschaffen.', 'satz': '', 'nummer': [], 'listenende': ''}


## Case study - resolving all citations from a court decision

A typical use case: you have extracted citation strings from a court decision
and want to resolve each one to its statutory text.

In [20]:
raw_citations = [
    "§ 242 BGB",
    "§ 433 Abs. 1 BGB",
    "§ 2 Abs. 1 Nr. 1 UrhG",
    "§§ 46 Abs. 2 ArbGG, 91 Abs. 1 ZPO",
]

for raw in raw_citations:
    results = lib.query(raw)
    for r in results:
        print(f"[{r.resolved_depth}] {str(r.reference)}")
        if r.titel():
            print(f"  {r.titel()}")
        print()

[absatz] § 242 BGB
  Leistung nach Treu und Glauben

[absatz] § 433 Abs. 1 BGB
  Vertragstypische Pflichten beim Kaufvertrag

[nummer] § 2 Abs. 1 Nr. 1 UrhG
  Geschützte Werke

[absatz] § 46 Abs. 2 ArbGG
  Grundsatz

[absatz] § 91 Abs. 1 ZPO
  Grundsatz und Umfang der Kostenpflicht



## Available laws

In [21]:
print(f"{lib.law_count} laws loaded")
print(lib.available_laws[:5])

6867 laws loaded
['1-DM-GOLDMÜNZG', '1. BESVNG', '1. BIMSCHV', '1. BMELDDÜV', '1. DV LUFTBO']
